# Separación de la categoría roja en dos fenómenos distintos

## Librerias

In [1]:
import json, re, unicodedata
from difflib import SequenceMatcher
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from pyproj import Transformer
import hashlib
from pyproj import Geod
import folium

# Localiza constantes.py sin rutas absolutas: sube desde el cuaderno
# buscando la raíz del proyecto (la que contiene "Bloque A") y lo busca
# primero en la raíz y luego dentro de Bloque A, que es donde vive hoy.
_aqui = Path.cwd().resolve()
_raiz = next(
    (p for p in [_aqui, *_aqui.parents] if (p / "Bloque A").is_dir()),
    None,
)
if _raiz is None:
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto (carpeta con 'Bloque A') "
        "subiendo desde " + str(_aqui)
    )

_mod = next(
    (c for c in (_raiz, _raiz / "Bloque A") if (c / "constantes.py").exists()),
    None,
)
if _mod is None:
    raise FileNotFoundError("No se encontró constantes.py en " + str(_raiz))

sys.path.insert(0, str(_mod))

import constantes

# BLOQUE B1 — Separación de la categoría roja en dos fenómenos distintos

In [2]:
# Correcciones:
#   - deduplicación de registros repetidos DENTRO de cada fuente
#   - vínculos secundarios N:1 (varios puntos OSM sobre un mismo sitio OCM)
#   - los seis casos dudosos entran como veredicto declarado, no como supuesto
# =============================================================================

RUTA_OUT = constantes.ruta_entregable("B1")

# Umbrales y CRS: se importan de constantes.py. No se redeclaran aquí para
# que no puedan existir dos copias que diverjan entre bloques.
UMBRAL_CONFIRMADA  = constantes.UMBRAL_CONFIRMADA   # 25 m — error típico de GPS urbano
UMBRAL_PROBABLE    = constantes.UMBRAL_PROBABLE     # 100 m — lado largo mediano de manzana (UAECD)
RADIO_BUSQUEDA_MAX = constantes.RADIO_BUSQUEDA_MAX  # 500 m — más allá no se postula pareja
RADIO_SECUNDARIO   = constantes.RADIO_SECUNDARIO    # 300 m — radio para vínculo N:1
RADIO_DUPLICADO    = constantes.RADIO_DUPLICADO     # 50 m — radio para duplicado interno
UMBRAL_TEXTUAL     = constantes.UMBRAL_TEXTUAL      # 0,45 — convención de similitud textual

CRS_GEO  = constantes.CRS_WGS84
CRS_PROY = constantes.CRS_METRICO

CASOS = {
    "C3-1": (302193, "node/13009696528"),   # Nicolás de Federmán
    "C3-2": (475140, "node/13010000994"),   # Universidad EAN
    "C3-3": (470939, "node/13009702030"),   # Terminal Salitre
    "N1-1": (311258, "way/58069884"),       # Compensar / Bolera
    "N1-2": (85062,  "node/13017542768"),   # Unicentro
    "N1-3": (470940, "node/13009700956"),   # San Andresito
}

# Los veredictos los produce el cuaderno B1_verificacion.ipynb.
# Si el archivo no existe, todos quedan en None y las categorías conservan el
# sufijo «(sin verificar)». B1 no depende de ese cuaderno para poder correr.
ARCHIVO_VEREDICTOS = RUTA_OUT / "B1_veredictos.json"
VEREDICTOS = {c: None for c in CASOS}

if ARCHIVO_VEREDICTOS.exists():
    with open(ARCHIVO_VEREDICTOS, encoding="utf-8") as fh:
        _leidos = json.load(fh)
    for _c in VEREDICTOS:
        if _c in _leidos.get("veredictos", {}):
            VEREDICTOS[_c] = _leidos["veredictos"][_c]
    print(f"Veredictos cargados ({_leidos.get('fecha_verificacion', '?')}): "
          f"{sum(v is not None for v in VEREDICTOS.values())} de {len(CASOS)}")
else:
    print("Sin archivo de veredictos. Los seis casos quedan sin verificar.")

_tr = Transformer.from_crs(CRS_GEO, CRS_PROY, always_xy=True)

print(f"Salidas en: {RUTA_OUT}")

Sin archivo de veredictos. Los seis casos quedan sin verificar.
Salidas en: F:\Tesis\Bloque B\Entregable B1


# PROCEDENCIA DE LOS INSUMOS

In [3]:
# =============================================================================
# PROCEDENCIA DE LOS INSUMOS
# =============================================================================
# B1 no descarga nada: consume lo que A2, A3 y A4 congelaron en disco. Si uno
# de esos archivos se regenera con otra fecha de corte, los resultados de B1
# cambian sin que nada lo advierta. Esta celda deja constancia de contra qué
# versión exacta se calculó, y falla si algo cambió desde la última ejecución.

INSUMOS = {
    "A2_Bogota":    constantes.ruta_entregable("A2", "A2_estaciones_OSM_Bogota.geojson"),
    "A2_Medellin":  constantes.ruta_entregable("A2", "A2_estaciones_OSM_Medellin.geojson"),
    "A3_Bogota":    constantes.ruta_entregable("A3", "A3_OpenChargeMap_Bogota_estaciones.csv"),
    "A3_Medellin":  constantes.ruta_entregable("A3", "A3_OpenChargeMap_Medellin_estaciones.csv"),
    "A4_resultado": constantes.ruta_entregable("A4", "A4_resultado_final_OCM_OSM.csv"),
}


def huella(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as fh:
        for trozo in iter(lambda: fh.read(65536), b""):
            h.update(trozo)
    return h.hexdigest()


procedencia = {}
for etiqueta, ruta in INSUMOS.items():
    constantes.exigir_archivo(ruta)          # falla claro si no existe
    procedencia[etiqueta] = {
        "archivo": ruta.name,
        "sha256": huella(ruta),
        "bytes": ruta.stat().st_size,
    }

# --- Encadenamiento con las huellas de los crudos congelados -----------------
# A2 y A3 ya registran la huella de sus archivos crudos. B1 las recoge para
# que el manifiesto documente la cadena completa: crudo -> derivado -> B1, y
# no solo el último eslabón.

cadena_crudos = {}

# A2: un .sha256 por ciudad, junto al crudo de Overpass
for archivo_hash in sorted(constantes.ruta_entregable("A2").glob("*.sha256")):
    cadena_crudos[archivo_hash.stem] = {
        "sha256": archivo_hash.read_text(encoding="utf-8").split()[0],
        "registrado_por": "A2",
    }

# A3: crudo congelado de OCM, identificado por su prefijo
for crudo in sorted(
    constantes.ruta_entregable("A3").glob(f"{constantes.PREFIJO_CRUDO_OCM}*.json")
):
    cadena_crudos[crudo.name] = {
        "sha256": huella(crudo),
        "registrado_por": "A3",
    }

# --- Comparación contra la ejecución anterior -------------------------------

MANIFIESTO = RUTA_OUT / "B1_procedencia_insumos.json"

if MANIFIESTO.exists():
    with open(MANIFIESTO, encoding="utf-8") as fh:
        _previo = json.load(fh)
    previo = _previo.get("insumos", {})
    previo_crudos = _previo.get("crudos", {})

    cambiados = [
        k for k in procedencia
        if k in previo and previo[k]["sha256"] != procedencia[k]["sha256"]
    ]
    crudos_cambiados = [
        k for k in cadena_crudos
        if k in previo_crudos
        and previo_crudos[k]["sha256"] != cadena_crudos[k]["sha256"]
    ]
    nuevos = [k for k in procedencia if k not in previo]

    if cambiados or crudos_cambiados:
        raise RuntimeError(
            "Los insumos cambiaron desde la última ejecución de B1. "
            f"Derivados: {cambiados or 'ninguno'}. "
            f"Crudos: {crudos_cambiados or 'ninguno'}. "
            "Revise si la regeneración fue intencional; si lo fue, borre "
            f"{MANIFIESTO.name} y vuelva a ejecutar."
        )

    print(f"Procedencia verificada: {len(procedencia) - len(nuevos)} derivados "
          f"y {len(cadena_crudos)} crudos sin cambios.")
    if nuevos:
        print(f"  Insumos nuevos, no comparables: {nuevos}")
else:
    print("Primera ejecución: se registra la huella de los insumos.")

# --- Registro ---------------------------------------------------------------

RUTA_OUT.mkdir(parents=True, exist_ok=True)
with open(MANIFIESTO, "w", encoding="utf-8") as fh:
    json.dump(
        {
            "fecha_corte_local": constantes.FECHA_CORTE_LOCAL,
            "fecha_corte_osm": constantes.FECHA_CORTE_OSM,
            "insumos": procedencia,
            "crudos": cadena_crudos,
        },
        fh, ensure_ascii=False, indent=2,
    )

print("\nDerivados que consume B1:")
for etiqueta, datos in procedencia.items():
    print(f"  {etiqueta:14s} {datos['sha256'][:12]}  {datos['bytes']:>9,d} bytes")

print("\nCrudos congelados encadenados:")
for nombre, datos in cadena_crudos.items():
    print(f"  {datos['registrado_por']:3s} {datos['sha256'][:12]}  {nombre}")

Primera ejecución: se registra la huella de los insumos.

Derivados que consume B1:
  A2_Bogota      99f132a15e2a     46,344 bytes
  A2_Medellin    69a8c0bafbf8     11,673 bytes
  A3_Bogota      cb12e624d5aa     14,303 bytes
  A3_Medellin    02767749ab5e      5,871 bytes
  A4_resultado   e9726767e7a6     13,157 bytes

Crudos congelados encadenados:
  A2  883195e669b3  A2_osm_crudo_Bogota_2026-08-29
  A2  109443ae81cd  A2_osm_crudo_Medellin_2026-08-29
  A3  c281a19abe3f  A3_ocm_crudo_Colombia_estado-al-2026-08-29_bajado-2026-09-02.json
  A3  ef448c305fde  A3_ocm_crudo_Colombia_estado-al-2026-08-29_bajado-2026-09-02.meta.json


# Utilidades de texto y proyección

In [4]:
# =============================================================================
# UTILIDADES DE TEXTO Y PROYECCIÓN
# =============================================================================
# La medida textual es la misma de A4, para que las dos etapas del
# emparejamiento midan lo mismo. Allí se midió que de 190 parejas candidatas,
# 15 alcanzan el umbral solo por la vía de cadenas; en 13 de ellas la
# distancia decide igual, y las 2 restantes —Unicentro y EnelX Modelia, ambas
# bajo 25 m— son coincidencias genuinas que Jaccard pierde porque las fuentes
# escriben el nombre propio de otra forma. De ahí que se conserve el max() y
# no se priorice Jaccard.

PALABRAS_NO_INFORMATIVAS = constantes.PALABRAS_NO_INFORMATIVAS


def normalizar(t):
    """Normaliza un texto para permitir comparaciones entre fuentes."""
    if t is None or (isinstance(t, float) and np.isnan(t)):
        return ""
    s = str(t).lower()
    s = "".join(c for c in unicodedata.normalize("NFD", s)
                if unicodedata.category(c) != "Mn")
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", s)).strip()


def obtener_tokens(valor):
    """Términos informativos de un texto, sin palabras de relleno."""
    texto = normalizar(valor)
    if not texto:
        return set()
    return {t for t in texto.split() if t not in PALABRAS_NO_INFORMATIVAS}


def similitud_tokens(a, b):
    """Jaccard entre los tokens informativos."""
    ta, tb = obtener_tokens(a), obtener_tokens(b)
    if not ta or not tb:
        return np.nan
    union = ta | tb
    if not union:
        return np.nan
    return len(ta & tb) / len(union)


def similitud_cadena(a, b):
    """Similitud de secuencia entre los textos normalizados."""
    na, nb = normalizar(a), normalizar(b)
    if not na or not nb:
        return np.nan
    return SequenceMatcher(None, na, nb).ratio()


def similitud(a, b):
    """Combina Jaccard y similitud de cadena; conserva el mayor.

    Devuelve NaN solo cuando ninguna de las dos vías pudo calcularse, es
    decir cuando falta el texto de alguno de los dos lados. Esa distinción
    importa: NaN significa «no comparable», no «no se parecen».
    """
    valores = [v for v in (similitud_tokens(a, b), similitud_cadena(a, b))
               if not pd.isna(v)]
    return max(valores) if valores else np.nan


def contencion_token(a, b):
    """True si los tokens informativos de un nombre están dentro del otro.

    Complementa a similitud(): 'Compensar' contra 'Punto de carga
    Compensar Avenida 68' da 0,41 por cadena y 0,17 por Jaccard, de modo
    que el max() tampoco lo rescata, y aun así es el mismo sitio.
    """
    ta = {t for t in obtener_tokens(a) if len(t) > 2}
    tb = {t for t in obtener_tokens(b) if len(t) > 2}
    if not ta or not tb:
        return False
    return ta <= tb or tb <= ta

def proyectar(lon, lat):
    x, y = _tr.transform(np.asarray(lon, float), np.asarray(lat, float))
    return np.asarray(x), np.asarray(y)

# Carga de fuentes

In [5]:
osm_bruto, ocm_bruto = constantes.cargar_fuentes_estaciones()

for df in (osm_bruto, ocm_bruto):
    df["x"], df["y"] = proyectar(df.lon, df.lat)

print(f"Brutos: OCM {len(ocm_bruto)} | OSM {len(osm_bruto)}")

Brutos: OCM 38 | OSM 53


# Deduplicación interna

In [6]:
# =============================================================================
# DEDUPLICACIÓN INTERNA
# =============================================================================
# Colapsa registros de la MISMA fuente que describen el mismo punto. Es un
# problema distinto del emparejamiento entre fuentes y debe resolverse antes:
# un duplicado interno no tratado se presenta después como una estación sin
# contraparte, inflando la categoría C4.
#
# Los grupos se resuelven por cierre transitivo. Si A se fusiona con B y B con
# C, los tres son el mismo sitio y debe quedar uno solo; tratando los pares de
# forma independiente, el registro de fusión podría señalar como conservado a
# uno que a su vez fue absorbido.

def deduplicar(df, etiqueta):
    fusiones, descartar = [], set()

    for ciudad in df.ciudad.unique():
        S = df[df.ciudad == ciudad]
        idx = S.index.to_numpy()
        D = np.hypot(S.x.values[:, None] - S.x.values[None, :],
                     S.y.values[:, None] - S.y.values[None, :])
        np.fill_diagonal(D, np.inf)

        # --- 1. pares candidatos: cercanos y con evidencia textual ---
        vinculos = []
        for a, b in zip(*np.where(np.triu(D <= RADIO_DUPLICADO))):
            ia, ib = idx[a], idx[b]
            na, nb = df.at[ia, "nombre"], df.at[ib, "nombre"]
            s = similitud(na, nb)
            if (pd.notna(s) and s >= UMBRAL_TEXTUAL) or contencion_token(na, nb):
                vinculos.append((ia, ib, D[a, b]))

        if not vinculos:
            continue

        # --- 2. cierre transitivo: componentes conexas ---
        padre = {}

        def raiz(k):
            padre.setdefault(k, k)
            while padre[k] != k:
                padre[k] = padre[padre[k]]
                k = padre[k]
            return k

        for ia, ib, _ in vinculos:
            ra, rb = raiz(ia), raiz(ib)
            if ra != rb:
                padre[rb] = ra

        grupos = {}
        for k in padre:
            grupos.setdefault(raiz(k), []).append(k)

        # --- 3. un superviviente por grupo ---
        for miembros in grupos.values():
            conservado = min(miembros, key=lambda k: str(df.at[k, "id"]))
            for absorbido in miembros:
                if absorbido == conservado:
                    continue
                descartar.add(absorbido)
                d = next((v for ia, ib, v in vinculos
                          if {ia, ib} == {conservado, absorbido}), np.nan)
                fusiones.append({
                    "fuente": etiqueta, "ciudad": ciudad,
                    "id_conservado": df.at[conservado, "id"],
                    "id_absorbido": df.at[absorbido, "id"],
                    "nombre_conservado": df.at[conservado, "nombre"],
                    "nombre_absorbido": df.at[absorbido, "nombre"],
                    "miembros_del_grupo": len(miembros),
                    "distancia_m": round(d, 1) if pd.notna(d) else np.nan,
                })

    return df.drop(index=descartar).reset_index(drop=True), pd.DataFrame(fusiones)


def duplicados_no_detectables(df, etiqueta):
    """Pares cercanos que la regla no puede evaluar por falta de nombre.

    La fusión exige evidencia textual, de modo que dos registros sin nombre
    a pocos metros nunca se detectan. No se fusionan a ciegas —serían
    puntos de carga distintos del mismo sitio, que es justamente lo que el
    vínculo N:1 modela—, pero su número debe reportarse: acota cuántos
    duplicados internos pueden haber sobrevivido.
    """
    casos = []
    for ciudad in df.ciudad.unique():
        S = df[df.ciudad == ciudad]
        idx = S.index.to_numpy()
        D = np.hypot(S.x.values[:, None] - S.x.values[None, :],
                     S.y.values[:, None] - S.y.values[None, :])
        np.fill_diagonal(D, np.inf)
        for a, b in zip(*np.where(np.triu(D <= RADIO_DUPLICADO))):
            ia, ib = idx[a], idx[b]
            na, nb = df.at[ia, "nombre"], df.at[ib, "nombre"]
            if not normalizar(na) or not normalizar(nb):
                casos.append({"fuente": etiqueta, "ciudad": ciudad,
                              "id_a": df.at[ia, "id"], "id_b": df.at[ib, "id"],
                              "nombre_a": na, "nombre_b": nb,
                              "distancia_m": round(D[a, b], 1)})
    return pd.DataFrame(casos)


sin_nombre = pd.concat(
    [duplicados_no_detectables(ocm_bruto, "OCM"),
     duplicados_no_detectables(osm_bruto, "OSM")],
    ignore_index=True)

osm, fus_osm = deduplicar(osm_bruto, "OSM")
ocm, fus_ocm = deduplicar(ocm_bruto, "OCM")
fusiones = pd.concat([fus_ocm, fus_osm], ignore_index=True)

print(f"Tras deduplicar: OCM {len(ocm)} | OSM {len(osm)}  ({len(fusiones)} fusiones)")
print(fusiones.to_string(index=False) if len(fusiones) else "(ninguna)")

print(f"\nPares cercanos no evaluables por falta de nombre: {len(sin_nombre)}")
if len(sin_nombre):
    print(sin_nombre.to_string(index=False))

Tras deduplicar: OCM 37 | OSM 52  (2 fusiones)
fuente   ciudad    id_conservado id_absorbido                   nombre_conservado                 nombre_absorbido  miembros_del_grupo  distancia_m
   OCM Medellin           485434       485435        Centro Comercial Los Molinos Centro Comercial Los Molinos EPM                   2         21.7
   OSM   Bogota node/13015491671 way/24587409 Punto de carga Compensar Avenida 68                        Compensar                   2          6.9

Pares cercanos no evaluables por falta de nombre: 0


# Emparejamiento primario mutuo 1:1

In [7]:
# Se acepta una pareja solo si cada registro es el mejor candidato del otro.
# La reciprocidad es lo que hace simétrico el resultado —emparejar OCM→OSM y
# OSM→OCM da el mismo conjunto— y lo que impide el defecto medido en A4, donde
# un mismo nodo de OSM era el vecino más cercano de cinco estaciones de OCM a
# la vez, en cada ciudad.

def emparejar(ocm, osm, radio=RADIO_BUSQUEDA_MAX):
    pares = []
    for ciudad in sorted(set(ocm.ciudad) | set(osm.ciudad)):
        # Los índices son LOCALES por ciudad: se reinician con reset_index,
        # de modo que mezclarlos entre ciudades uniría filas equivocadas.
        A = ocm[ocm.ciudad == ciudad].reset_index(drop=True)
        B = osm[osm.ciudad == ciudad].reset_index(drop=True)
        if A.empty or B.empty:
            continue
        D = np.hypot(A.x.values[:, None] - B.x.values[None, :],
                     A.y.values[:, None] - B.y.values[None, :])
        mej_a, mej_b = D.argmin(axis=1), D.argmin(axis=0)
        # La condición mej_b[j] == i es la reciprocidad. Garantiza 1:1 por sí
        # sola: si i eligió a j y j eligió a i, ninguno puede formar otra
        # pareja, de modo que no hace falta llevar conjuntos de consumo.
        for i, j in enumerate(mej_a):
            if mej_b[j] != i or D[i, j] > radio:
                continue
            pares.append({"ciudad": ciudad,
                          "ocm_id": A.at[i, "id"], "osm_uid": B.at[j, "id"],
                          "nombre_ocm": A.at[i, "nombre"], "nombre_osm": B.at[j, "nombre"],
                          "operador_ocm": A.at[i, "operador"],
                          "operador_osm": B.at[j, "operador"],
                          "distancia_m": D[i, j]})

    P = pd.DataFrame(pares)
    if P.empty:
        return P

    P["similitud_nombre"] = [similitud(a, b) for a, b in zip(P.nombre_ocm, P.nombre_osm)]
    P["similitud_operador"] = [similitud(a, b) for a, b in zip(P.operador_ocm, P.operador_osm)]
    # NaN significa «no comparable», no «no se parecen»: la guarda explícita
    # evita que una comparación imposible cuente como evidencia.
    P["evidencia_textual"] = [
        (pd.notna(sn) and sn >= UMBRAL_TEXTUAL) or (pd.notna(so) and so >= UMBRAL_TEXTUAL)
        for sn, so in zip(P.similitud_nombre, P.similitud_operador)]
    return P


pares = emparejar(ocm, osm)
print(f"Pares primarios: {len(pares)}")

# Verificación independiente: el mutuo debe ser simétrico. Emparejar en el
# sentido contrario tiene que producir exactamente el mismo conjunto.
_inv = emparejar(osm, ocm, radio=RADIO_BUSQUEDA_MAX)
_a = set(zip(pares.ocm_id, pares.osm_uid))
_b = set(zip(_inv.osm_uid, _inv.ocm_id))
print(f"Simetría OCM→OSM vs OSM→OCM: "
      f"{'OK' if _a == _b else 'FALLA'} ({len(_a & _b)} de {len(_a)})")

Pares primarios: 20
Simetría OCM→OSM vs OSM→OCM: OK (20 de 20)


# Vínculos secundarios N:1

In [8]:
# Las dos fuentes no cuentan lo mismo: OSM registra puntos de carga
# individuales y OCM registra el sitio agregado. Un centro comercial con tres
# cargadores puede ser tres nodos en OSM y un solo registro en OCM.
#
# Los nodos de OSM que quedan sin pareja pero caen cerca de un sitio de OCM ya
# emparejado no son vacío de cobertura: son puntos adicionales del mismo
# lugar. Contarlos como C4 «solo OSM» sobreestimaría la falta de cobertura de
# OCM, que es justo la conclusión que el trabajo quiere medir bien.

def vinculos_secundarios(ocm, osm, pares, radio=RADIO_SECUNDARIO):
    v = []
    ocm_emp = set(pares.ocm_id) if len(pares) else set()
    osm_emp = set(pares.osm_uid) if len(pares) else set()

    for ciudad in sorted(set(ocm.ciudad)):
        A = ocm[(ocm.ciudad == ciudad) & ocm.id.isin(ocm_emp)].reset_index(drop=True)
        B = osm[(osm.ciudad == ciudad) & ~osm.id.isin(osm_emp)].reset_index(drop=True)
        if A.empty or B.empty:
            continue
        D = np.hypot(A.x.values[:, None] - B.x.values[None, :],
                     A.y.values[:, None] - B.y.values[None, :])
        for j in range(len(B)):
            i = D[:, j].argmin()
            if D[i, j] <= radio:
                v.append({"ciudad": ciudad, "ocm_id": A.at[i, "id"],
                          "osm_uid": B.at[j, "id"], "nombre_ocm": A.at[i, "nombre"],
                          "nombre_osm": B.at[j, "nombre"],
                          "operador_ocm": A.at[i, "operador"],
                          "operador_osm": B.at[j, "operador"],
                          "distancia_m": D[i, j]})

    V = pd.DataFrame(v)
    if V.empty:
        return V

    # El vínculo se postula por proximidad, pero la evidencia textual es lo
    # que permite distinguir un punto adicional del mismo sitio de un vecino
    # que simplemente está cerca. Con 300 m de radio, en zona urbana densa,
    # la distancia sola no basta.
    V["similitud_nombre"] = [similitud(a, b) for a, b in zip(V.nombre_ocm, V.nombre_osm)]
    V["similitud_operador"] = [similitud(a, b) for a, b in zip(V.operador_ocm, V.operador_osm)]
    V["evidencia_textual"] = [
        (pd.notna(sn) and sn >= UMBRAL_TEXTUAL) or (pd.notna(so) and so >= UMBRAL_TEXTUAL)
        for sn, so in zip(V.similitud_nombre, V.similitud_operador)]
    return V


secundarios = vinculos_secundarios(ocm, osm, pares)
print(f"Vínculos secundarios: {len(secundarios)}")

# Diagnóstico para B3: si el punto secundario se parece más al sitio de OCM
# que el registro emparejado como primario, la asignación puede estar
# invertida. No se corrige aquí —la decisión es de B3, con el score
# combinado— pero debe quedar señalado.
if len(secundarios):
    _p = pares.set_index("ocm_id")[["osm_uid", "distancia_m", "similitud_nombre"]]
    revisar = []
    for _, f in secundarios.iterrows():
        if f.ocm_id not in _p.index:
            continue
        prim = _p.loc[f.ocm_id]
        if (pd.notna(f.similitud_nombre)
                and prim.distancia_m > UMBRAL_CONFIRMADA
                and (pd.isna(prim.similitud_nombre)
                     or f.similitud_nombre > prim.similitud_nombre)):
            revisar.append({
                "ocm_id": f.ocm_id, "nombre_ocm": f.nombre_ocm,
                "primario": prim.osm_uid, "d_primario": round(prim.distancia_m, 1),
                "sim_primario": prim.similitud_nombre,
                "secundario": f.osm_uid, "d_secundario": round(f.distancia_m, 1),
                "sim_secundario": f.similitud_nombre})
    revisar = pd.DataFrame(revisar)
    print(f"\nAsignaciones a revisar en B3 (el secundario se parece más): {len(revisar)}")
    if len(revisar):
        print(revisar.to_string(index=False))

Vínculos secundarios: 3

Asignaciones a revisar en B3 (el secundario se parece más): 2
ocm_id          nombre_ocm         primario  d_primario  sim_primario       secundario  d_secundario  sim_secundario
470940 EnelX San Andresito node/13010048854        82.5      0.117647 node/13009700956         120.2        0.517241
311258           Compensar node/13015491671        43.0      0.409091     way/58069884         299.9        0.720000


# Veredictos y clasificación

In [9]:
# Seis parejas no se resuelven con la regla automática y quedan declaradas
# como casos. Mientras no exista veredicto, la categoría lleva el sufijo
# «(sin verificar)»: el cuaderno no supone la respuesta, la deja marcada.

def veredicto_de(ocm_id, osm_uid):
    for caso, (o, s) in CASOS.items():
        if str(o) == str(ocm_id) and s == osm_uid:
            return caso, VEREDICTOS[caso]
    return None, None


for tabla in (pares, secundarios):
    if len(tabla):
        tabla["caso"], tabla["veredicto"] = zip(
            *[veredicto_de(a, b) for a, b in zip(tabla.ocm_id, tabla.osm_uid)])
    else:
        tabla["caso"] = pd.Series(dtype="object")
        tabla["veredicto"] = pd.Series(dtype="object")

# Los seis casos están escritos a mano. Si un identificador deja de existir
# —porque la fuente cambió o la deduplicación absorbió el registro— el
# veredicto se vuelve un no-op silencioso y la pareja se clasifica sola.
_ubicados = set(pares.caso.dropna()) | set(secundarios.caso.dropna())
if set(CASOS) - _ubicados:
    raise AssertionError(
        f"Casos declarados que no aparecen en el resultado: "
        f"{sorted(set(CASOS) - _ubicados)}. Revise si el identificador "
        "cambió o si el registro fue absorbido en la deduplicación."
    )


def clasificar(f):
    """Clasifica una pareja primaria.

    Por debajo del umbral probable decide la distancia. Por encima, la
    pareja solo se sostiene si hay respaldo: un veredicto manual afirmativo
    o evidencia textual. Sin ninguno de los dos se degrada, porque una
    pareja lejana y sin parecido no es desacuerdo posicional entre dos
    registros de la misma estación: son dos estaciones distintas, y
    llamarla desacuerdo inflaría ese diagnóstico.
    """
    d = f.distancia_m
    if d <= UMBRAL_CONFIRMADA:
        return "C1 Coincidencia confirmada"
    if d <= UMBRAL_PROBABLE:
        return "C2 Coincidencia probable"
    if f.veredicto is False:
        return "DEGRADADO"
    if f.veredicto is True:
        return "C3 Desacuerdo posicional"
    if f.evidencia_textual:
        return "C3 Desacuerdo posicional (sin verificar)"
    return "DEGRADADO"


pares["categoria"] = pares.apply(clasificar, axis=1) if len(pares) else pd.Series(dtype="object")
degradados = pares[pares.categoria == "DEGRADADO"].copy()
pares_val = pares[pares.categoria != "DEGRADADO"].copy()


def clasificar_sec(f):
    """Clasifica un vínculo secundario.

    Mismo criterio que arriba: la proximidad postula el vínculo, pero con
    300 m de radio hace falta respaldo para afirmarlo. Sin veredicto ni
    evidencia textual el vínculo no se sostiene y el nodo vuelve a contarse
    como presente solo en OpenStreetMap.
    """
    if f.veredicto is True:
        return "C5 Punto secundario del mismo sitio"
    if f.veredicto is False:
        return None          # sitio distinto -> vuelve a C4 solo OSM
    if f.evidencia_textual:
        return "C5 Punto secundario del mismo sitio (sin verificar)"
    return None


if len(secundarios):
    secundarios["categoria"] = secundarios.apply(clasificar_sec, axis=1)
else:
    secundarios["categoria"] = pd.Series(dtype="object")

sec_val = secundarios[secundarios.categoria.notna() &
                      secundarios.osm_uid.isin(
                          set(osm.id) - set(pares_val.osm_uid))].copy()

print(f"Pares válidos: {len(pares_val)}  |  degradados: {len(degradados)}")
print(f"Vínculos secundarios válidos: {len(sec_val)} de {len(secundarios)}")

Pares válidos: 17  |  degradados: 3
Vínculos secundarios válidos: 3 de 3


# Inventario unificado

In [10]:
# Una fila por estación. Las cuatro categorías que exige el entregable son
# C1 a C4; C5 y DEGRADADO son refinamientos propios que se colapsan sobre
# ellas al reportar (celda de resultados).
#
# Los registros de parejas degradadas no se pierden: al no entrar en
# pares_val, cada lado vuelve a contarse por separado como C4, que es lo
# correcto cuando la pareja no se sostiene.

usados_ocm = set(pares_val.ocm_id)
usados_osm = set(pares_val.osm_uid) | set(sec_val.osm_uid)

reg = []
for _, f in pares_val.iterrows():
    reg.append({"ciudad": f.ciudad, "categoria": f.categoria, "origen": "AMBAS",
                "ocm_id": f.ocm_id, "osm_uid": f.osm_uid, "nombre_ocm": f.nombre_ocm,
                "nombre_osm": f.nombre_osm, "distancia_m": round(f.distancia_m, 1),
                "similitud_nombre": f.similitud_nombre,
                "similitud_operador": f.similitud_operador, "caso": f.caso})

for _, f in sec_val.iterrows():
    reg.append({"ciudad": f.ciudad, "categoria": f.categoria, "origen": "OSM->OCM",
                "ocm_id": f.ocm_id, "osm_uid": f.osm_uid, "nombre_ocm": f.nombre_ocm,
                "nombre_osm": f.nombre_osm, "distancia_m": round(f.distancia_m, 1),
                "similitud_nombre": f.similitud_nombre,
                "similitud_operador": f.similitud_operador, "caso": f.caso})

for _, f in ocm[~ocm.id.isin(usados_ocm)].iterrows():
    reg.append({"ciudad": f.ciudad, "categoria": "C4 Sin contraparte (solo OCM)",
                "origen": "OCM", "ocm_id": f.id, "osm_uid": None,
                "nombre_ocm": f.nombre, "nombre_osm": None, "distancia_m": np.nan,
                "similitud_nombre": np.nan, "similitud_operador": np.nan, "caso": None})

for _, f in osm[~osm.id.isin(usados_osm)].iterrows():
    reg.append({"ciudad": f.ciudad, "categoria": "C4 Sin contraparte (solo OSM)",
                "origen": "OSM", "ocm_id": None, "osm_uid": f.id,
                "nombre_ocm": None, "nombre_osm": f.nombre, "distancia_m": np.nan,
                "similitud_nombre": np.nan, "similitud_operador": np.nan, "caso": None})

B1 = pd.DataFrame(reg)

# Los identificadores son etiquetas, no cantidades. Con nulos presentes
# pandas infiere float y escribe '475409.0', lo que rompe las uniones contra
# A3 y A4 —de hecho hizo fallar una verificación de la tutoría—. El tipo
# 'string' admite nulos sin degradar a float.
for col in ("ocm_id", "osm_uid"):
    B1[col] = B1[col].astype("string")

B1 = B1.sort_values(["ciudad", "categoria", "distancia_m"]).reset_index(drop=True)

# Estaciones físicas: una fila C5 es un punto adicional de un sitio ya
# contado, no una estación más. Es la cifra que va al informe.
estaciones = B1[~B1.categoria.str.startswith("C5")]
print(f"Registros en el inventario: {len(B1)}")
print(f"Estaciones físicas: {len(estaciones)}  "
      f"(Bogotá {(estaciones.ciudad == 'Bogota').sum()}, "
      f"Medellín {(estaciones.ciudad == 'Medellin').sum()})")

# Verificación: todo registro de cada fuente debe estar representado.
assert B1.ocm_id.nunique() == len(ocm), "Falta alguna estación de OCM"
assert B1.osm_uid.nunique() == len(osm), "Falta alguna estación de OSM"

Registros en el inventario: 72
Estaciones físicas: 69  (Bogotá 50, Medellín 19)


# Pruebas de aceptación

In [11]:
_geod = Geod(ellps="GRS80")   # mismo elipsoide que MAGNA-SIRGAS (EPSG:9377)


def verificar_distancias(tabla, tolerancia_rel=0.001, piso_m=0.05):
    """Recalcula cada distancia sobre el elipsoide y la compara con la
    reportada, que vino de proyectar a EPSG:9377.

    Son dos caminos distintos —geodésica directa contra proyección plana—
    de modo que un desalineamiento de índices o un intercambio de
    coordenadas produce diferencias de cientos o miles de metros. Es la
    falla que el Bloque A no detectó: las distancias reportadas se veían
    razonables y estaban mal.

    Los dos métodos no coinciden exactamente, y no deben hacerlo. Toda
    proyección plana deforma, y EPSG:9377 comprime las distancias de forma
    proporcional: medido sobre estas parejas, unas 620 partes por millón en
    Bogotá y 190 en Medellín. La diferencia entre ciudades responde a su
    distancia al meridiano central de la proyección. Sobre el umbral de
    25 m eso equivale a 1,5 cm, sin efecto sobre la clasificación.

    De ahí la tolerancia: 0,1 % —por encima del factor de escala medido,
    muy por debajo del error que la prueba debe detectar— con piso de 5 cm
    para las parejas muy cortas, donde el término proporcional se vuelve
    despreciable.

    Las coordenadas se recuperan por identificador desde las fuentes, no se
    toman de la tabla de parejas: así la prueba no hereda el eventual error
    de unión que busca detectar.
    """
    if not len(tabla):
        return True, pd.DataFrame()

    c = tabla[["ocm_id", "osm_uid", "distancia_m"]].copy()
    c = c.merge(ocm[["id", "lat", "lon"]].rename(
        columns={"id": "ocm_id", "lat": "lat_ocm", "lon": "lon_ocm"}),
        on="ocm_id", how="left")
    c = c.merge(osm[["id", "lat", "lon"]].rename(
        columns={"id": "osm_uid", "lat": "lat_osm", "lon": "lon_osm"}),
        on="osm_uid", how="left")

    if c[["lat_ocm", "lat_osm"]].isna().any().any():
        raise AssertionError(
            "Hay parejas cuyas coordenadas no se pudieron recuperar: el "
            "identificador reportado no existe en la fuente."
        )

    _, _, geodesica = _geod.inv(
        c.lon_ocm.values, c.lat_ocm.values,
        c.lon_osm.values, c.lat_osm.values)

    dif = np.abs(geodesica - c.distancia_m.values)
    tol = np.maximum(piso_m, c.distancia_m.values * tolerancia_rel)

    detalle = pd.DataFrame({
        "ocm_id": c.ocm_id, "osm_uid": c.osm_uid,
        "reportada_m": c.distancia_m.round(2),
        "geodesica_m": np.round(geodesica, 2),
        "diferencia_m": dif.round(3),
        "tolerancia_m": tol.round(3),
        "pasa": dif <= tol})
    return bool(detalle.pasa.all()), detalle


def verificar_vecino_mas_cercano():
    """Comprueba que el registro unido sea el más cercano disponible.

    Es la segunda falla del Bloque A: la distancia puede estar bien
    calculada y aun así unirse la fila equivocada. Se admite excepción
    cuando el vecino más cercano ya fue tomado por otra pareja mutua.
    """
    fallas = []
    for ciudad in sorted(set(pares_val.ciudad)):
        A = ocm[ocm.ciudad == ciudad].set_index("id")
        B = osm[osm.ciudad == ciudad].reset_index(drop=True)
        tomados = set(pares_val[pares_val.ciudad == ciudad].osm_uid)
        for _, f in pares_val[pares_val.ciudad == ciudad].iterrows():
            a = A.loc[f.ocm_id]
            d = np.hypot(a.x - B.x.values, a.y - B.y.values)
            for k in np.argsort(d):
                cand = B.at[k, "id"]
                if cand == f.osm_uid:
                    break
                if cand not in tomados:
                    fallas.append({"ocm_id": f.ocm_id, "unido": f.osm_uid,
                                   "mas_cercano_libre": cand,
                                   "d_unido": round(f.distancia_m, 1),
                                   "d_cercano": round(d[k], 1)})
                    break
    return len(fallas) == 0, pd.DataFrame(fallas)


def pruebas():
    ok_d, det_dist = verificar_distancias(pares)
    ok_v, det_vec = verificar_vecino_mas_cercano()

    r = [
        ("Distancia verificada por camino independiente (geodésica GRS80)",
         ok_d,
         f"{int(det_dist.pasa.sum())} de {len(det_dist)}; dif. máx. "
         f"{det_dist.diferencia_m.max():.3f} m" if len(det_dist) else "sin parejas"),
        ("El registro unido es el más cercano disponible",
         ok_v, "sin excepciones" if ok_v else f"{len(det_vec)} casos"),
        ("Conservación OCM: toda estación figura en el inventario",
         B1.ocm_id.nunique() == len(ocm), f"{B1.ocm_id.nunique()} de {len(ocm)}"),
        ("Conservación OSM: toda estación figura en el inventario",
         B1.osm_uid.nunique() == len(osm), f"{B1.osm_uid.nunique()} de {len(osm)}"),
        ("Ningún registro de OSM aparece dos veces",
         B1.osm_uid.dropna().duplicated().sum() == 0,
         f"{int(B1.osm_uid.dropna().duplicated().sum())} duplicados"),
        ("OCM repetido solo por vínculo N:1",
         set(B1[B1.ocm_id.duplicated(keep=False)].dropna(subset=["ocm_id"]).categoria)
         <= {"C1 Coincidencia confirmada", "C2 Coincidencia probable",
             "C3 Desacuerdo posicional", "C3 Desacuerdo posicional (sin verificar)",
             "C5 Punto secundario del mismo sitio",
             "C5 Punto secundario del mismo sitio (sin verificar)"},
         "solo categorías de pareja o secundario"),
        ("La deduplicación no perdió registros",
         len(osm) + len(fus_osm) == len(osm_bruto)
         and len(ocm) + len(fus_ocm) == len(ocm_bruto),
         f"OSM {len(osm)}+{len(fus_osm)}={len(osm_bruto)}; "
         f"OCM {len(ocm)}+{len(fus_ocm)}={len(ocm_bruto)}"),
        ("Los seis casos declarados aparecen en el resultado",
         set(CASOS) <= set(B1.caso.dropna()),
         f"{B1.caso.nunique()} de {len(CASOS)}"),
    ]

    for nombre, ok, detalle in r:
        print(f"  [{'OK   ' if ok else 'FALLA'}] {nombre}  ({detalle})")

    if not ok_d:
        print("\nParejas que no pasaron la verificación de distancia:")
        print(det_dist[~det_dist.pasa].to_string(index=False))
    if not ok_v:
        print("\nParejas cuyo vecino más cercano libre es otro:")
        print(det_vec.to_string(index=False))

    resumen = pd.DataFrame(
        [{"prueba": n, "resultado": "PASÓ" if o else "FALLÓ", "detalle": d}
         for n, o, d in r])
    return all(o for _, o, _ in r), resumen, det_dist


todas_pasan, resumen_pruebas, detalle_distancias = pruebas()

  [OK   ] Distancia verificada por camino independiente (geodésica GRS80)  (20 de 20; dif. máx. 0.287 m)
  [OK   ] El registro unido es el más cercano disponible  (sin excepciones)
  [OK   ] Conservación OCM: toda estación figura en el inventario  (37 de 37)
  [OK   ] Conservación OSM: toda estación figura en el inventario  (52 de 52)
  [OK   ] Ningún registro de OSM aparece dos veces  (0 duplicados)
  [OK   ] OCM repetido solo por vínculo N:1  (solo categorías de pareja o secundario)
  [OK   ] La deduplicación no perdió registros  (OSM 52+1=53; OCM 37+1=38)
  [OK   ] Los seis casos declarados aparecen en el resultado  (6 de 6)


# Factor de escala de la proyección

In [12]:
# El docstring de verificar_distancias afirma un factor de escala de unas
# 620 ppm en Bogotá y 190 en Medellín. Este es el cálculo que lo sustenta:
# un dato citado en un comentario debe tener su medición visible.

d = detalle_distancias.merge(
    pares[["ocm_id", "ciudad"]], on="ocm_id", how="left")
d["rel_ppm"] = (d.diferencia_m / d.reportada_m * 1e6).round(0)

print(d.sort_values("reportada_m")[
    ["ciudad", "reportada_m", "geodesica_m", "diferencia_m", "rel_ppm"]
].to_string(index=False))

print("\nFactor de escala medido por ciudad (ppm):")
print(d.groupby("ciudad").rel_ppm.agg(["mean", "min", "max", "count"])
      .round(0).to_string())
print("\nLa proporción constante dentro de cada ciudad confirma que la "
      "diferencia es deformación de la proyección y no error de cálculo.\n"
      "La diferencia entre ciudades responde a su distancia al meridiano "
      "central de EPSG:9377.")

  ciudad  reportada_m  geodesica_m  diferencia_m  rel_ppm
  Bogota         1.72         1.72         0.001    581.0
  Bogota        10.16        10.17         0.007    689.0
Medellin        15.76        15.76         0.003    190.0
  Bogota        15.90        15.91         0.010    629.0
  Bogota        23.55        23.56         0.014    594.0
  Bogota        27.54        27.56         0.018    654.0
  Bogota        28.19        28.20         0.018    639.0
  Bogota        30.48        30.50         0.018    591.0
  Bogota        31.96        31.98         0.020    626.0
  Bogota        41.16        41.19         0.026    632.0
  Bogota        43.03        43.06         0.027    627.0
Medellin        46.86        46.85         0.009    192.0
  Bogota        79.50        79.54         0.049    616.0
  Bogota        82.51        82.57         0.051    618.0
  Bogota       138.98       139.07         0.086    619.0
  Bogota       167.28       167.39         0.106    634.0
  Bogota      

# Resultados

In [13]:
print("--- Conteos absolutos, seis categorías ---")
t = B1.groupby(["categoria", "ciudad"]).size().unstack(fill_value=0)
t["TOTAL"] = t.sum(axis=1)
print(t.to_string())

# El entregable pide cuatro categorías. C5 y DEGRADADO son refinamientos
# propios: el primero marca un punto adicional de un sitio ya contado, el
# segundo una pareja que no se sostiene y cuyos dos lados vuelven a contarse
# por separado. Se colapsan para poder leer el resultado contra lo pedido.
print("\n--- Conteos absolutos, cuatro categorías del entregable ---")
CANONICA = {
    "C1 Coincidencia confirmada": "Coincidencia confirmada",
    "C2 Coincidencia probable": "Coincidencia probable",
    "C3 Desacuerdo posicional": "Desacuerdo posicional",
    "C3 Desacuerdo posicional (sin verificar)": "Desacuerdo posicional",
    "C4 Sin contraparte (solo OCM)": "Presente solo en OpenChargeMap",
    "C4 Sin contraparte (solo OSM)": "Presente solo en OpenStreetMap",
}
# Las filas C5 no son estaciones: se excluyen del conteo por categoría.
canon = B1[~B1.categoria.str.startswith("C5")].copy()
canon["categoria_entregable"] = canon.categoria.map(CANONICA)
assert canon.categoria_entregable.notna().all(), "Categoría sin equivalencia canónica"

tc = canon.groupby(["categoria_entregable", "ciudad"]).size().unstack(fill_value=0)
tc["TOTAL"] = tc.sum(axis=1)
print(tc.to_string())
print(f"\nEstaciones físicas: {len(canon)}  "
      f"(Bogotá {(canon.ciudad == 'Bogota').sum()}, "
      f"Medellín {(canon.ciudad == 'Medellin').sum()})")
print(f"Puntos adicionales del mismo sitio (C5), no contados como estación: "
      f"{(B1.categoria.str.startswith('C5')).sum()}")

# --- Comparación con el método unidireccional de A4 -------------------------
# A4 y B1 no responden la misma pregunta: A4 recorre solo OpenChargeMap y
# produce una fila por estación de esa fuente; B1 recorre las dos y produce
# una fila por estación real. Los porcentajes de A4 están calculados sobre su
# propio denominador, de modo que la comparación es de cobertura, no de
# números equivalentes.
print("\n--- A4 unidireccional frente a B1 bidireccional ---")
a4 = pd.read_csv(
    constantes.ruta_entregable("A4", "A4_resultado_final_OCM_OSM.csv"),
    encoding="utf-8-sig",
    dtype={"ocm_id": "string", "osm_uid": "string"},
)
# Defensa: si el archivo se generó antes de fijar el tipo, los ids traen '.0'
# y ninguna unión cruzaría, fallando en silencio.
for col in ("ocm_id", "osm_uid"):
    if col in a4.columns:
        a4[col] = a4[col].str.replace(r"\.0$", "", regex=True)

osm_evaluadas_a4 = a4.osm_uid.dropna().nunique()
comparacion = pd.DataFrame([
    {"metodo": "A4 unidireccional", "registros": len(a4),
     "base": "solo estaciones de OCM",
     "osm_sin_evaluar": len(osm_bruto) - osm_evaluadas_a4},
    {"metodo": "B1 bidireccional", "registros": len(B1),
     "base": "inventario conjunto", "osm_sin_evaluar": 0},
])
print(comparacion.to_string(index=False))

print("\n--- Descomposición del semáforo de A4 ---")
m = B1[B1.ocm_id.notna()].drop_duplicates("ocm_id").set_index("ocm_id")["categoria"]
sem = a4[["ocm_id", "semaforo"]].copy()
sem["categoria_B1"] = sem.ocm_id.map(m).fillna("Absorbido por deduplicación interna")
cruzados = sem.categoria_B1.ne("Absorbido por deduplicación interna").sum()
print(f"Registros de A4 localizados en B1: {cruzados} de {len(sem)}")
if cruzados == 0:
    raise AssertionError(
        "Ningún identificador de A4 cruzó contra B1. Revise el tipo de "
        "ocm_id: es el síntoma del '.0' en los identificadores."
    )
descomposicion_semaforo = pd.crosstab(sem.semaforo, sem.categoria_B1)
print(descomposicion_semaforo.to_string())

print("\n--- Casos declarados, pendientes de verificación manual ---")
p = B1[B1.caso.notna()]
print(p[["caso", "categoria", "ocm_id", "osm_uid", "nombre_ocm",
         "nombre_osm", "distancia_m"]].to_string(index=False))

print("\n--- Degradados (pareja lejana sin respaldo textual) ---")
print(degradados[["ciudad", "ocm_id", "nombre_ocm", "nombre_osm",
                  "distancia_m", "similitud_nombre"]].to_string(index=False)
      if len(degradados) else "(ninguno)")

--- Conteos absolutos, seis categorías ---
ciudad                                               Bogota  Medellin  TOTAL
categoria                                                                   
C1 Coincidencia confirmada                                4         1      5
C2 Coincidencia probable                                  8         1      9
C3 Desacuerdo posicional (sin verificar)                  3         0      3
C4 Sin contraparte (solo OCM)                            12         8     20
C4 Sin contraparte (solo OSM)                            23         9     32
C5 Punto secundario del mismo sitio (sin verificar)       3         0      3

--- Conteos absolutos, cuatro categorías del entregable ---
ciudad                          Bogota  Medellin  TOTAL
categoria_entregable                                   
Coincidencia confirmada              4         1      5
Coincidencia probable                8         1      9
Desacuerdo posicional                3         0      3


# VERIFICACIÓN MANUAL DE LOS SEIS CASOS

In [14]:
# =============================================================================
# MAPA DE LOS SEIS CASOS PENDIENTES DE VERIFICACIÓN
# =============================================================================
# Los seis casos no se resuelven con la regla automática: están lejos pero
# tienen nombres compatibles. Aceptarlos por el nombre sería repetir el
# criterio automático con otra etiqueta, de modo que se deciden mirando si
# los dos puntos caen sobre el mismo predio.

_c = pd.concat([pares, secundarios], ignore_index=True)
_c = _c[_c.caso.notna()].copy()
_c = _c.merge(ocm[["id", "lat", "lon"]].rename(
    columns={"id": "ocm_id", "lat": "lat_ocm", "lon": "lon_ocm"}),
    on="ocm_id", how="left")
_c = _c.merge(osm[["id", "lat", "lon"]].rename(
    columns={"id": "osm_uid", "lat": "lat_osm", "lon": "lon_osm"}),
    on="osm_uid", how="left")

mapa = folium.Map(
    location=[_c.lat_ocm.mean(), _c.lon_ocm.mean()], zoom_start=12,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/"
          "World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery")

# Un color por caso, para poder referirse a ellos sin ambigüedad cuando
# varios caen en la misma zona de la ciudad.
COLOR_CASO = {
    "C3-1": "#00FFFF",   # cian
    "C3-2": "#FFFF00",   # amarillo
    "C3-3": "#FF6D00",   # naranja
    "N1-1": "#FF00FF",   # magenta
    "N1-2": "#00FF3C",   # verde lima
    "N1-3": "#FF1744",   # rojo intenso
}

for _, f in _c.iterrows():
    color = COLOR_CASO.get(f.caso, "#FFFFFF")
    grupo = folium.FeatureGroup(name=f"{f.caso} — {f.nombre_ocm}", show=True)

    # Halo oscuro detrás de cada elemento, para que el color se lea tanto
    # sobre asfalto claro como sobre vegetación oscura.
    folium.PolyLine(
        [[f.lat_ocm, f.lon_ocm], [f.lat_osm, f.lon_osm]],
        color="#000000", weight=8, opacity=0.6).add_to(grupo)
    folium.PolyLine(
        [[f.lat_ocm, f.lon_ocm], [f.lat_osm, f.lon_osm]],
        color=color, weight=4, opacity=1, dash_array="10,6",
        tooltip=f"{f.caso}: {f.distancia_m:.0f} m").add_to(grupo)

    # OpenChargeMap — círculo relleno
    folium.CircleMarker(
        [f.lat_ocm, f.lon_ocm], radius=11, color="#000000", weight=5,
        fill=False, opacity=0.6).add_to(grupo)
    folium.CircleMarker(
        [f.lat_ocm, f.lon_ocm], radius=10, color=color, weight=3,
        fill=True, fill_color=color, fill_opacity=0.9,
        popup=f"<b>{f.caso} · OCM {f.ocm_id}</b><br>{f.nombre_ocm}",
        tooltip=f"OCM · {f.caso}").add_to(grupo)

    # OpenStreetMap — círculo hueco, mismo color del caso
    folium.CircleMarker(
        [f.lat_osm, f.lon_osm], radius=11, color="#000000", weight=6,
        fill=False, opacity=0.6).add_to(grupo)
    folium.CircleMarker(
        [f.lat_osm, f.lon_osm], radius=10, color=color, weight=4,
        fill=True, fill_color="#000000", fill_opacity=0.35,
        popup=f"<b>{f.caso} · OSM {f.osm_uid}</b><br>{f.nombre_osm}",
        tooltip=f"OSM · {f.caso}").add_to(grupo)

    grupo.add_to(mapa)

# Leyenda fija sobre el mapa
_filas = "".join(
    f'<div style="margin:2px 0"><span style="display:inline-block;width:12px;'
    f'height:12px;background:{COLOR_CASO[c]};border:1px solid #000;'
    f'margin-right:6px"></span>{c}</div>'
    for c in COLOR_CASO if c in set(_c.caso))

mapa.get_root().html.add_child(folium.Element(f"""
<div style="position:fixed; bottom:24px; left:24px; z-index:9999;
     background:rgba(255,255,255,.92); padding:10px 12px; border-radius:6px;
     font:12px sans-serif; box-shadow:0 1px 6px rgba(0,0,0,.4)">
  <b>Casos</b>
  {_filas}
  <hr style="margin:6px 0">
  <div>● relleno = OpenChargeMap</div>
  <div>○ hueco = OpenStreetMap</div>
</div>"""))

folium.LayerControl(collapsed=False).add_to(mapa)
mapa.save(RUTA_OUT / "B1_casos_verificacion.html")

print("Mapa en:", RUTA_OUT / "B1_casos_verificacion.html")
print("Un color por caso. Círculo relleno = OpenChargeMap, hueco = OpenStreetMap.")
print("Para cada caso: ¿los dos puntos caen sobre el mismo predio o complejo?\n")
print(_c[["caso", "ocm_id", "nombre_ocm", "osm_uid", "nombre_osm",
          "distancia_m"]].to_string(index=False))

Mapa en: F:\Tesis\Bloque B\Entregable B1\B1_casos_verificacion.html
Un color por caso. Círculo relleno = OpenChargeMap, hueco = OpenStreetMap.
Para cada caso: ¿los dos puntos caen sobre el mismo predio o complejo?

caso ocm_id                         nombre_ocm          osm_uid                                     nombre_osm  distancia_m
C3-2 475140        Parqueadero universidad EAN node/13010000994                       Universidad EAN (Bogotá)   183.067643
C3-3 470939             EnelX Terminal Salitre node/13009702030 Punto de carga Terminal de Transportes Salitre   295.131518
C3-1 302193          EnelX Nicolas de Federman node/13009696528     Punto de carga Clínica Nicolás de Federmán   138.984294
N1-3 470940                EnelX San Andresito node/13009700956        Punto de carga San Andresito Carrera 38   120.187214
N1-2  85062 Centro comercial Unicentro Bogotá  node/13017542768                       Punto de carga Unicentro   250.880906
N1-1 311258                          Comp

# Guardado

In [15]:
RUTA_OUT.mkdir(parents=True, exist_ok=True)

salidas = {
    # Resultado principal
    "B1_clasificacion_cuatro_categorias.csv": (B1, False),
    "B1_conteos_absolutos.csv": (t, True),
    "B1_conteos_cuatro_categorias.csv": (tc, True),

    # Trazabilidad del método
    "B1_pares_primarios.csv": (pares, False),
    "B1_fusiones_internas.csv": (fusiones, False),
    "B1_vinculos_secundarios.csv": (secundarios, False),
    "B1_degradados.csv": (degradados, False),

    # Verificación
    "B1_pruebas_aceptacion.csv": (resumen_pruebas, False),
    "B1_verificacion_distancias.csv": (detalle_distancias, False),

    # Comparación con el método anterior
    "B1_comparacion_con_A4.csv": (comparacion, False),
    "B1_descomposicion_semaforo_A4.csv": (descomposicion_semaforo, True),
}

for nombre, (tabla, con_indice) in salidas.items():
    tabla.to_csv(RUTA_OUT / nombre, index=con_indice, encoding="utf-8-sig")
    print(f"  {nombre:45s} {len(tabla):>4d} filas")

print(f"\nGuardado en: {RUTA_OUT}")
print(f"Pruebas de aceptación: {'TODAS PASAN' if todas_pasan else 'HAY FALLAS'}")

  B1_clasificacion_cuatro_categorias.csv          72 filas
  B1_conteos_absolutos.csv                         6 filas
  B1_conteos_cuatro_categorias.csv                 5 filas
  B1_pares_primarios.csv                          20 filas
  B1_fusiones_internas.csv                         2 filas
  B1_vinculos_secundarios.csv                      3 filas
  B1_degradados.csv                                3 filas
  B1_pruebas_aceptacion.csv                        8 filas
  B1_verificacion_distancias.csv                  20 filas
  B1_comparacion_con_A4.csv                        2 filas
  B1_descomposicion_semaforo_A4.csv                3 filas

Guardado en: F:\Tesis\Bloque B\Entregable B1
Pruebas de aceptación: TODAS PASAN
